# Projeto 01 — Agente de Rotas para Comida Local (IA 2026-27)

**Inteligência Artificial — Licenciatura em Engenharia de Sistemas Informáticos (IPCA / EST)**

### Contexto
Uma plataforma digital liga **produtores** locais, **clientes** e **centros de recolha (armazéns)**.
Cada cliente faz uma encomenda e escolhe um **local de entrega**. Os produtos são recolhidos nos produtores,
passam obrigatoriamente pelo armazém (`producer > warehouse > delivery_location`) e são depois entregues,
respeitando:
* a **janela de disponibilidade** do produto no produtor (`availability_start / availability_end`);
* a **janela de entrega** no local de destino (`delivery_start / delivery_end`);
* a **capacidade** dos veículos.

O objetivo do projeto é desenvolver um agente que escolha as melhores rotas de recolha e entrega usando
**pesquisa informada** (Greedy, A*), **pesquisa local** (Hill-Climbing) e **CSP** (biblioteca `python-constraint`).

### Equipa (Grupo G##)
| Nome | Número |
|------|--------|
| ... | ... |
| ... | ... |
| ... | ... |
| ... | ... |

### Este notebook — Entrega 1: Parser e resumo dos dados (`01_eda_tiny.ipynb`)
1. **Parser** — lê os ficheiros CSV do dataset e carrega-os para variáveis Python, seguindo a mesma lógica do
   notebook `Notebook_91_CSP_CourseTT_PC_w_data.ipynb`: uma função `parse_...` devolve um dicionário
   `parsed_data` com uma secção por tipo de dados; depois imprimem-se as secções para verificação e
   constroem-se as estruturas auxiliares que os algoritmos vão usar.
2. **Resumo dos dados importados** — quantos produtos e de que tipo, produtores, veículos, locais de entrega,
   encomendas, etc.

A análise completa, com gráficos, está no notebook `01_eda_tiny_comp.ipynb`.

In [ ]:
# Instalar as bibliotecas usadas no projeto
# (python-constraint2 é a versão mantida da biblioteca python-constraint: importa-se com "from constraint import *")
%pip install pandas python-constraint2

In [ ]:
# Importar bibliotecas
import glob
import os
import zipfile
from datetime import datetime

import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

---
## 1. Escolha do dataset

Existem dois datasets (fornecidos no Moodle em ZIP):
* `tiny`  — pequeno e simples;
* `small` — maior e mais complexo.

Basta alterar a variável `DATASET` para correr todo o notebook sobre o outro dataset.
Se a pasta ainda não existir, o ZIP correspondente é extraído automaticamente para a pasta `data/`.

In [ ]:
DATASET = 'tiny'      # 'tiny' ou 'small'
DATA_DIR = 'data'

def get_dataset_folder(dataset, data_dir=DATA_DIR):
    # Return the folder of the dataset, extracting the ZIP file if needed
    folders = glob.glob(os.path.join(data_dir, f'data_{dataset}_*'))
    if not folders:
        zips = glob.glob(f'data_{dataset}_*.zip')
        if not zips:
            raise FileNotFoundError(f'ZIP do dataset "{dataset}" não encontrado')
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(data_dir)
        folders = glob.glob(os.path.join(data_dir, f'data_{dataset}_*'))
    return sorted(folders)[0]

folder = get_dataset_folder(DATASET)
print('Pasta do dataset:', folder)
print(sorted(os.listdir(folder)))

---
## 2. Parser

Tal como no notebook de referência (`parse_file` → `parsed_data`), a função `parse_dataset`:
1. lê cada ficheiro CSV com `pandas.read_csv` para uma **tabela** (`DataFrame`), já com as datas convertidas;
2. guarda as tabelas no dicionário `tables` (útil para os resumos);
3. converte cada tabela num dicionário indexado pela **chave primária (PK)** e guarda-o em `parsed_data`
   (útil para os algoritmos: `parsed_data['products'][3]` devolve o produto com `product_id = 3`).

| Secção | Ficheiro | Chave | Conteúdo |
|---|---|---|---|
| `rules` | rules.csv | `rule` | parâmetros do problema (horizonte, tempos de serviço, ...) |
| `localities` | localities.csv | `locality_id` | freguesias geocodificadas |
| `producers` | producers.csv | `producer_id` | produtores |
| `products` | products.csv | `product_id` | produtos de cada produtor |
| `centers` | collection_centers.csv | `center_id` | armazéns (base dos veículos) |
| `vehicles` | vehicles.csv | `vehicle_id` | frota |
| `customers` | customers.csv | `customer_id` | clientes |
| `delivery_locations` | delivery_locations.csv | `delivery_location_id` | locais de entrega |
| `orders` | orders.csv | `order_id` | encomendas |
| `order_items` | order_items.csv | `order_item_id` | linhas de encomenda com as janelas temporais |
| `locations` | locations.csv | `location_id` | paragens de routing (produtores, armazém, entregas) |
| `distance_matrix` | distance_matrix.csv | `(origem, destino)` | distância (km) e duração (min) |

In [ ]:
# Information about each CSV file: section -> (file name, primary key, columns with dates)
FILES = {
    'localities':         ('localities.csv',         'locality_id',          []),
    'producers':          ('producers.csv',          'producer_id',          []),
    'products':           ('products.csv',           'product_id',           []),
    'centers':            ('collection_centers.csv', 'center_id',            []),
    'vehicles':           ('vehicles.csv',           'vehicle_id',           []),
    'customers':          ('customers.csv',          'customer_id',          []),
    'delivery_locations': ('delivery_locations.csv', 'delivery_location_id', []),
    'orders':             ('orders.csv',             'order_id',             ['order_datetime']),
    'order_items':        ('order_items.csv',        'order_item_id',
                           ['availability_start', 'availability_end', 'delivery_start', 'delivery_end']),
    'locations':          ('locations.csv',          'location_id',          []),
    'distance_matrix':    ('distance_matrix.csv',    None,                   []),
}


def parse_rule_value(value):
    # Convert a rule value to int or float when possible, otherwise keep the text
    try:
        return int(value)
    except ValueError:
        pass
    try:
        return float(value)
    except ValueError:
        return value


def parse_dataset(folder):
    # Function to read and parse all the CSV files of a dataset
    tables = {}        # section -> pandas DataFrame
    data = {}          # section -> {primary key: record}

    # Rules: rule -> value
    rules_df = pd.read_csv(os.path.join(folder, 'rules.csv'), dtype=str)
    data['rules'] = {}
    for rule, value in zip(rules_df['rule'], rules_df['value']):
        data['rules'][rule] = parse_rule_value(value)
    tables['rules'] = rules_df

    # Entity tables
    for section, (file_name, pk, date_columns) in FILES.items():
        df = pd.read_csv(os.path.join(folder, file_name), parse_dates=date_columns)
        if section == 'vehicles':
            # The column "refrigerated" is 1/0 in the file: convert it to True/False
            df['refrigerated'] = df['refrigerated'].astype(bool)
        tables[section] = df
        if pk is None:
            continue
        if df[pk].duplicated().any():
            raise ValueError(f'{file_name}: chave primária repetida')
        # DataFrame -> dictionary {primary key: {column: value}}
        data[section] = df.set_index(pk, drop=False).to_dict('index')

    # Distance matrix: (origin, destination) -> {'distance_km': ..., 'duration_min': ...}
    data['distance_matrix'] = {}
    for row in tables['distance_matrix'].itertuples():
        key = (row.origin_location_id, row.destination_location_id)
        data['distance_matrix'][key] = {'distance_km': row.distance_km, 'duration_min': row.duration_min}

    return data, tables


parsed_data, tables = parse_dataset(folder)
print({section: len(records) for section, records in parsed_data.items()})

In [ ]:
# Output the parsed data for verification
print('Rules:')
for rule, value in parsed_data['rules'].items():
    print(f'  {rule}: {value!r}')

print('\nProducers (producer_id: name @ locality -> products):')
for pid, p in parsed_data['producers'].items():
    prods = [pr['name'] for pr in parsed_data['products'].values() if pr['producer_id'] == pid]
    loc = parsed_data['localities'][p['locality_id']]
    print(f"  {pid}: {p['name']} @ {loc['name']} ({loc['municipality']}) -> {prods}")

print('\nCollection centers:')
for cid, c in parsed_data['centers'].items():
    print(f"  {cid}: {c['name']} @ locality {c['locality_id']}, capacity {c['capacity_kg']} kg")

print('\nVehicles:')
for vid, v in parsed_data['vehicles'].items():
    print(f"  {vid}: {v['plate']} center={v['center_id']} capacity={v['capacity_kg']} kg "
          f"refrigerated={v['refrigerated']} speed={v['avg_speed_kmh']} km/h")

print('\nDelivery locations:')
for did, d in parsed_data['delivery_locations'].items():
    print(f"  {did}: {d['name']} ({d['location_type']}) @ locality {d['locality_id']}")

print('\nOrders (order_id: customer -> delivery location, n items, kg):')
for oid, o in parsed_data['orders'].items():
    items = [it for it in parsed_data['order_items'].values() if it['order_id'] == oid]
    print(f"  {oid}: customer {o['customer_id']} -> DL {o['delivery_location_id']}, "
          f"{len(items)} items, {sum(it['weight_kg'] for it in items):.1f} kg")

print('\nFirst 3 order items:')
for it in list(parsed_data['order_items'].values())[:3]:
    print(' ', it)

print('\nDistance matrix (first 3 pairs):')
for key, val in list(parsed_data['distance_matrix'].items())[:3]:
    print(f'  {key}: {val}')

### 2.1 Estruturas auxiliares para os algoritmos

Tal como no notebook de referência (onde se constroem `weekcls`, `course_list`, `lesson_list`, `study_plan`),
transformamos os dados carregados nas estruturas que os algoritmos de pesquisa e o CSP vão usar:

* **Tempo em minutos**: todas as datas são convertidas em minutos desde `start_date 00:00`
  (dia 1 = 0..1439, dia 2 = 1440..2879). Assim, as janelas temporais passam a ser intervalos de inteiros.
* **Paragens (`location_id`)**: cada produtor, armazém e local de entrega corresponde a uma paragem de
  `locations.csv`; as tabelas `loc_of_producer`, `loc_of_center` e `loc_of_delivery` fazem esta tradução.
* **`dist` / `dur`**: dicionários `(origem, destino) -> km / min` para consulta rápida.
* **`requests`**: um pedido de transporte por linha de encomenda (recolha no produtor → armazém → entrega).
* **`vehicles`**: capacidade e paragem de partida/chegada (armazém) de cada veículo.

In [ ]:
# Planning horizon and rules
rules = parsed_data['rules']
T0 = datetime.strptime(str(rules['start_date']), '%Y-%m-%d')   # time origin (minute 0)
HORIZON_MIN = rules['horizon_days'] * 24 * 60
SERVICE_MIN = rules['stop_service_min']            # service time at each pickup / delivery stop
HANDLING_MIN = rules['warehouse_handling_min']     # unload / load time at the warehouse
MAX_DWELL_MIN = rules['max_warehouse_dwell_h'] * 60

def to_min(dt):
    # Minutes since the beginning of the planning horizon
    return int((dt - T0).total_seconds() // 60)

def fmt_min(m):
    # Minutes since T0 -> 'D1 08:30'
    d, r = divmod(int(m), 1440)
    return f'D{d + 1} {r // 60:02d}:{r % 60:02d}'

# Routing stops: entity id -> location_id
loc_of_producer, loc_of_center, loc_of_delivery = {}, {}, {}
for lid, loc in parsed_data['locations'].items():
    {'producer': loc_of_producer,
     'collection_center': loc_of_center,
     'delivery_location': loc_of_delivery}[loc['location_type']][loc['ref_id']] = lid

stops = sorted(parsed_data['locations'])
dist = {k: v['distance_km'] for k, v in parsed_data['distance_matrix'].items()}
dur = {k: v['duration_min'] for k, v in parsed_data['distance_matrix'].items()}
for s in stops:                         # travelling to the same stop costs nothing
    dist[(s, s)] = dur[(s, s)] = 0.0

# Vehicles: capacity and depot (warehouse stop)
vehicles = {vid: {'capacity': v['capacity_kg'],
                  'depot': loc_of_center[v['center_id']],
                  'refrigerated': v['refrigerated']}
            for vid, v in parsed_data['vehicles'].items()}

# Requests: one per order item (pickup at producer -> warehouse -> delivery location)
requests = {}
for iid, it in parsed_data['order_items'].items():
    order = parsed_data['orders'][it['order_id']]
    product = parsed_data['products'][it['product_id']]
    requests[iid] = {
        'order': it['order_id'],
        'product': product['name'],
        'producer': product['producer_id'],
        'pickup': loc_of_producer[product['producer_id']],
        'delivery': loc_of_delivery[order['delivery_location_id']],
        'weight': it['weight_kg'],
        'av_start': to_min(it['availability_start']),
        'av_end': to_min(it['availability_end']),
        'del_start': to_min(it['delivery_start']),
        'del_end': to_min(it['delivery_end']),
    }

# Requests grouped by order and by producer
order_requests = {}
producer_requests = {}
for iid, r in requests.items():
    order_requests.setdefault(r['order'], []).append(iid)
    producer_requests.setdefault(r['producer'], []).append(iid)

WAREHOUSE = loc_of_center[1]
print('Paragens:', stops, '| armazém =', WAREHOUSE)
print('Produtor -> paragem:', loc_of_producer)
print('Local de entrega -> paragem:', loc_of_delivery)
print('Veículos:', vehicles)
print('Encomenda -> itens:', order_requests)
print('Produtor -> itens:', producer_requests)
print('\nPrimeiros pedidos:')
for iid in list(requests)[:5]:
    r = requests[iid]
    print(f"  {iid}: {r['product']:<10} {r['weight']:5.1f} kg  P{r['pickup']}->W{WAREHOUSE}->D{r['delivery']}  "
          f"disp [{fmt_min(r['av_start'])}, {fmt_min(r['av_end'])}]  "
          f"entrega [{fmt_min(r['del_start'])}, {fmt_min(r['del_end'])}]")

---
## 3. Resumo dos dados importados

Nesta secção resumimos, com `pandas`, o que foi carregado pelo parser (usando as tabelas em `tables`).

### 3.1 Regras do problema

In [ ]:
tables['rules']

### 3.2 Quantidades

In [ ]:
# Number of records of each type
summary = pd.DataFrame({
    'Quantidade': [
        len(tables['producers']),
        len(tables['products']),
        tables['products']['category'].nunique(),
        len(tables['centers']),
        len(tables['vehicles']),
        len(tables['customers']),
        len(tables['delivery_locations']),
        len(tables['orders']),
        len(tables['order_items']),
        len(tables['locations']),
        len(tables['distance_matrix']),
    ]
}, index=[
    'Produtores',
    'Produtos',
    'Categorias de produtos',
    'Armazéns (centros de recolha)',
    'Veículos',
    'Clientes',
    'Locais de entrega',
    'Encomendas',
    'Linhas de encomenda (itens)',
    'Paragens de routing',
    'Pares na matriz de distâncias',
])
summary

### 3.3 Produtos

Cada produto pertence a um produtor e tem uma categoria (`fruit`, `vegetable`, `other`) e uma unidade de venda
(`kg`, `unit`, `crate`). O peso de uma unidade está em `unit_weight_kg`.

In [ ]:
products = tables['products']

# How many products of each category and of each unit
print('Produtos por categoria:')
print(products['category'].value_counts().to_string())
print('\nProdutos por unidade de venda:')
print(products['unit'].value_counts().to_string())

# Names of the products of each category (without repetitions)
print('\nProdutos de cada categoria:')
for category, group in products.groupby('category'):
    names = sorted(group['name'].unique())
    print(f'  {category} ({len(group)} produtos, {len(names)} nomes diferentes): {", ".join(names)}')

In [ ]:
# Products sold by more than one producer
n_producers = products.groupby('name')['producer_id'].nunique()
print('Produtos vendidos por mais do que um produtor:')
print(n_producers[n_producers > 1].sort_values(ascending=False).to_string())

### 3.4 Produtores

In [ ]:
# Producers with their municipality and number of products of each category
producers = tables['producers'].merge(
    tables['localities'][['locality_id', 'municipality']], on='locality_id')
per_category = pd.crosstab(products['producer_id'], products['category'])

producers_summary = producers.set_index('producer_id')[['name', 'municipality']].join(per_category)
producers_summary['total produtos'] = per_category.sum(axis=1)
producers_summary

### 3.5 Armazém e veículos

In [ ]:
centers = tables['centers'].merge(tables['localities'][['locality_id', 'municipality']], on='locality_id')
print('Armazém(ns):')
print(centers[['center_id', 'name', 'municipality', 'capacity_kg']].to_string(index=False))

vehicles_df = tables['vehicles']
print('\nVeículos:')
print(vehicles_df[['vehicle_id', 'plate', 'center_id', 'capacity_kg', 'refrigerated', 'avg_speed_kmh']].to_string(index=False))
print(f"\nCapacidade total da frota: {vehicles_df['capacity_kg'].sum():.0f} kg")
print(f"Veículos refrigerados    : {int(vehicles_df['refrigerated'].sum())} de {len(vehicles_df)}")

### 3.6 Locais de entrega e clientes

In [ ]:
delivery = tables['delivery_locations'].merge(
    tables['localities'][['locality_id', 'municipality']], on='locality_id')
print('Locais de entrega por tipo:')
print(delivery['location_type'].value_counts().to_string())
print()
print(delivery[['delivery_location_id', 'name', 'location_type', 'municipality']].to_string(index=False))

orders = tables['orders']
orders_per_customer = orders['customer_id'].value_counts()
print(f'\nClientes: {len(tables["customers"])}')
print(f'Clientes com mais de uma encomenda: {(orders_per_customer > 1).sum()}')

### 3.7 Encomendas e itens

Cada encomenda tem um ou mais itens (linhas de encomenda). Cada item é um produto com uma quantidade e um peso.

In [ ]:
order_items = tables['order_items']

# Number of items and total weight of each order
orders_summary = order_items.groupby('order_id').agg(
    itens=('order_item_id', 'count'),
    peso_kg=('weight_kg', 'sum'))
orders_summary = orders[['order_id', 'customer_id', 'delivery_location_id']].set_index('order_id').join(orders_summary)

print(f"Encomendas: {len(orders)}  |  Itens: {len(order_items)}")
print(f"Itens por encomenda (mín / média / máx): {orders_summary['itens'].min()} / "
      f"{orders_summary['itens'].mean():.1f} / {orders_summary['itens'].max()}")
print(f"Peso total a transportar: {order_items['weight_kg'].sum():.1f} kg")
print(f"Peso por item (mín / média / máx): {order_items['weight_kg'].min()} / "
      f"{order_items['weight_kg'].mean():.1f} / {order_items['weight_kg'].max()} kg")
orders_summary

In [ ]:
# Items and weight per delivery day
per_day = order_items.groupby(order_items['delivery_start'].dt.date).agg(
    itens=('order_item_id', 'count'),
    peso_kg=('weight_kg', 'sum'))
per_day.index.name = 'dia de entrega'
print('Período de disponibilidade nos produtores:',
      order_items['availability_start'].min(), '->', order_items['availability_end'].max())
print('Período de entrega aos clientes          :',
      order_items['delivery_start'].min(), '->', order_items['delivery_end'].max())
per_day

### 3.8 Matriz de distâncias

In [ ]:
dm = tables['distance_matrix']
n_stops = len(tables['locations'])
print(f'Paragens: {n_stops}  ->  pares esperados: {n_stops * (n_stops - 1)}  |  pares no ficheiro: {len(dm)}')
print('\nParagens por tipo:')
print(tables['locations']['location_type'].value_counts().to_string())
print()
dm[['distance_km', 'duration_min']].describe().round(1)